# CropHarvest: EDA

CropHarvest is the crop / non-crop benchmark. One sample is a single field's
12-month time series of Sentinel-2, Sentinel-1 and climate, with a binary
`is_crop` label and a source `dataset` (the geographic group we hold out).

## Setup

In [ ]:
from pathlib import Path
import glob, json
import numpy as np
import matplotlib.pyplot as plt

# find the repo root by walking up to the folder that has data/
REPO = Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent
print('repo:', REPO)

CH = REPO / 'data' / 'input' / 'cropharvest'
ARRAYS = CH / 'features' / 'arrays'
files = sorted(glob.glob(str(ARRAYS / '*.h5')))
print(len(files), 'CropHarvest field files')

## Raw layout of one sample

Each `.h5` holds one `(12, 18)` array: 12 months by 18 raw bands. The columns:

| idx | 0 | 1 | 2..9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| band | VV | VH | B2..B8A | B9 | B11 | B12 | temp | precip | elevation | slope | NDVI |

In [ ]:
import h5py
# pick one field and read its raw (12, 18) array
path = files[0]
with h5py.File(path, 'r') as f:
    raw = np.asarray(f['array'], dtype=np.float32)
idx, dataset = Path(path).stem.split('_', 1)
print('file:', Path(path).name, '| shape:', raw.shape)

# look up this field's label + lat/lon in labels.geojson
feats = json.loads((CH / 'labels.geojson').read_text())['features']
label_of = {(int(g['properties']['index']), g['properties']['dataset']): g['properties'] for g in feats}
p = label_of[(int(idx), dataset)]
print('dataset:', dataset, '| is_crop:', p['is_crop'], '| lat/lon:', round(p['lat'], 3), round(p['lon'], 3))

## What this one field looks like over the year

The four modalities live on different scales, so each gets its own panel.

In [ ]:
months = list(range(1, 13))
fig, ax = plt.subplots(4, 1, figsize=(9, 11), sharex=True)

# Sentinel-2 reflectance (cols 2..9, 11, 12)
s2_cols = [2, 3, 4, 5, 6, 7, 8, 9, 11, 12]
s2_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
for c, n in zip(s2_cols, s2_names):
    ax[0].plot(months, raw[:, c], marker='o', label=n)
ax[0].set_title('Sentinel-2 reflectance'); ax[0].set_ylabel('reflectance'); ax[0].legend(ncol=5, fontsize=8)

# NDVI (col 17), its own [-1, 1] scale
ax[1].plot(months, raw[:, 17], marker='o', color='green')
ax[1].set_title('NDVI (greenness)'); ax[1].set_ylabel('NDVI'); ax[1].axhline(0, color='grey', lw=0.6)

# Sentinel-1 radar (cols 0, 1) in dB
ax[2].plot(months, raw[:, 0], marker='o', label='VV')
ax[2].plot(months, raw[:, 1], marker='o', label='VH')
ax[2].set_title('Sentinel-1 backscatter'); ax[2].set_ylabel('dB'); ax[2].legend()

# climate: temperature + precipitation (cols 13, 14) on twin axes
ax[3].plot(months, raw[:, 13], marker='o', color='tomato', label='temperature')
ax[3].set_ylabel('temperature', color='tomato')
axb = ax[3].twinx()
axb.plot(months, raw[:, 14], marker='s', color='steelblue', label='precipitation')
axb.set_ylabel('precipitation', color='steelblue')
ax[3].set_title('climate'); ax[3].set_xlabel('month'); ax[3].set_xticks(months)

fig.suptitle(f"one {dataset} field  (is_crop={p['is_crop']})")
plt.tight_layout(); plt.show()

## A few fields side by side: crop vs non-crop

NDVI trajectory is the clearest single signal: crops green up and senesce, bare
/ non-crop land stays flatter. We grab a handful of each and overlay them.

In [ ]:
crop_paths, noncrop_paths = [], []
for path in files:
    i, ds = Path(path).stem.split('_', 1)
    key = (int(i), ds)
    if key not in label_of:
        continue
    bucket = crop_paths if label_of[key]['is_crop'] == 1 else noncrop_paths
    if len(bucket) < 6:
        bucket.append(path)
    if len(crop_paths) >= 6 and len(noncrop_paths) >= 6:
        break

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for path in crop_paths:
    with h5py.File(path, 'r') as f:
        ax[0].plot(months, np.asarray(f['array'])[:, 17], alpha=0.7)
for path in noncrop_paths:
    with h5py.File(path, 'r') as f:
        ax[1].plot(months, np.asarray(f['array'])[:, 17], alpha=0.7)
ax[0].set_title('crop fields: NDVI'); ax[1].set_title('non-crop: NDVI')
ax[0].set_xlabel('month'); ax[1].set_xlabel('month'); ax[0].set_ylabel('NDVI')
plt.tight_layout(); plt.show()

## Dataset-level: label balance and source regions

In [ ]:
import collections
is_crop = [g['properties']['is_crop'] for g in feats]
datasets = [g['properties']['dataset'] for g in feats]
by_dataset = collections.Counter(datasets)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
vals, counts = np.unique(is_crop, return_counts=True)
ax[0].bar(['non-crop', 'crop'], counts, color=['#94a3b8', '#16a34a'])
ax[0].set_title(f'label balance (N={len(is_crop)})')
top = by_dataset.most_common(12)
ax[1].barh([k for k, _ in top][::-1], [v for _, v in top][::-1], color='#2563eb')
ax[1].set_title('samples per source dataset (top 12)')
plt.tight_layout(); plt.show()

## Geography

Source datasets are geographically clustered, which is exactly why strict
holdout (train on some regions, test on a held-out one) is a hard, realistic test.

In [ ]:
lat = np.array([g['properties']['lat'] for g in feats])
lon = np.array([g['properties']['lon'] for g in feats])
crop = np.array(is_crop)
plt.figure(figsize=(11, 5))
plt.scatter(lon[crop == 0], lat[crop == 0], s=3, c='#94a3b8', label='non-crop', alpha=0.4)
plt.scatter(lon[crop == 1], lat[crop == 1], s=3, c='#16a34a', label='crop', alpha=0.4)
plt.xlabel('longitude'); plt.ylabel('latitude'); plt.legend(); plt.title('CropHarvest sample locations')
plt.show()

## Takeaways

: one sample is a 12-step multi-sensor pixel time series, label = `is_crop`.
: NDVI carries most of the visible crop signal; S1 and climate add context.
: source `dataset` is the geographic holdout group (strict transfer).